In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score 
from sklearn.metrics import ConfusionMatrixDisplay 

In [ ]:
matches = pd.read_csv('../data/processed/features_v2.csv')

In [ ]:
matches['GoalDiffLast5'] = matches['HomeGoalsLast5'] - matches['AwayGoalsLast5'] 
matches['PointsDiffLast5'] = matches['HomePointsLast5'] - matches['AwayPointsLast5'] 
matches['GoalAgainstDiffLast5'] = matches['AwayGoalsAgainstLast5'] - matches['HomeGoalsAgainstLast5'] 
matches['ShotDiffLast5'] = matches['HomeShotsForLast5'] - matches['AwayShotsForLast5'] 
matches['ShotOTDiffLast5'] = matches['HomeShotsOnTargetLast5'] - matches['AwayShotsOnTargetLast5']

In [ ]:
features = [ 'PPGDiff', 'GDPerGameDiff', 'GoalsAgainstPerGameDiff', 'ShotOTDiffLast5', 'HomeAwayPointsDiffLast5', 'HomeAwayGoalsDiffLast5', 'HomeAwayGoalsAgainstDiffLast5' ]

In [ ]:
train = matches[matches['Season'] != '25-26'] 
test = matches[matches['Season'] == '25-26']

In [ ]:
X_train = train[features] 
y_train = train['FTR'] 
X_test = test[features] 
y_test = test['FTR']

In [ ]:
 
model = LogisticRegression(
    max_iter=1000,
)
model.fit(X_train, y_train)

In [ ]:
preds = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, preds) 
print(accuracy)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds)

In [ ]:
probs = model.predict_proba(X_test) 
print(model.classes_) 
print(probs[0])

In [ ]:
results = test[['HomeTeam', 'AwayTeam', 'FTR']].copy() 
results['Predicted'] = preds 
results['AwayProb'] = probs[:, 0] 
results['DrawProb'] = probs[:, 1] 
results['HomeProb'] = probs[:, 2] 
results.head(10)

In [ ]:
importance = pd.Series( model.coef_[2], index=features ).sort_values(ascending=False) 
print(importance)

In [ ]:
from sklearn.metrics import log_loss 
print(log_loss(y_test, probs))

In [ ]:
probs = model.predict_proba(X_test) 
results['DrawProb'] = probs[:, list(model.classes_).index('D')] 
results.sort_values('DrawProb', ascending=False).head(20)